In [ ]:
import urllib.request
import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt


In [ ]:

# ==========================================
# 1. Download the Data
# ==========================================

os.makedirs('data', exist_ok=True)

url  = 'https://storage.googleapis.com/quickdraw_dataset/full/simplified/cat.ndjson'
path = 'data/cat.ndjson'
if not os.path.exists(path):
    print('Downloading...')
    urllib.request.urlretrieve(url, path)
    print('Done.')
else:
    print('Already downloaded.')



In [ ]:

# ==========================================
# 2. Inspect the Raw Format
# ==========================================

drawings = []
with open('data/cat.ndjson') as f:
    for line in f:
        drawings.append(json.loads(line))
print(f'Total drawings: {len(drawings)}')
print()

d = drawings[0]
print('Keys         :', list(d.keys()))
print('Word         :', d['word'])
print('Recognised   :', d['recognized'])
print('Num strokes  :', len(d['drawing']))
print()
print('First stroke, first 5 x values:', d['drawing'][0][0][:5])
print('First stroke, first 5 y values:', d['drawing'][0][1][:5])

# TODO: Print the number of strokes in drawings[5] and drawings[10].
#       Are they the same? What does a higher stroke count mean visually?
strokes_5 = len(drawings[5]['drawing'])
strokes_10 = len(drawings[10]['drawing'])
print(f"Strokes in drawings[5]: {strokes_5}")
print(f"Strokes in drawings[10]: {strokes_10}")

# OBSERVATIONS:
# 1. The stroke counts are typically different because every person draws doodles uniquely.
# 2. Visually, a higher stroke count means the person lifted their pen off the canvas 
#    more frequently, creating a more detailed drawing or sketching multiple isolated parts 
#    (e.g., drawing whiskers, eyes, and ears as distinct line segments rather than one continuous line).



In [ ]:

# ==========================================
# 3. Converting to Stroke-3 Format
# ==========================================

def drawing_to_stroke3(drawing):
    '''
    Convert a Quick Draw drawing to stroke-3 format.
    Input  : drawing -- list of strokes, each stroke = [[x0,x1,...],[y0,y1,...]]
    Output : numpy array shape (total_points, 3), columns = [dx, dy, pen_lifted]
    '''
    strokes = []
    for stroke in drawing:
        xs, ys = stroke[0], stroke[1]
        for i in range(len(xs)):
            dx = xs[i] - xs[i-1] if i > 0 else 0
            dy = ys[i] - ys[i-1] if i > 0 else 0
            pen_lifted = 1 if i == len(xs) - 1 else 0
            strokes.append([dx, dy, pen_lifted])
    return np.array(strokes, dtype=np.float32)

s3 = drawing_to_stroke3(drawings[0]['drawing'])
print('Stroke-3 shape:', s3.shape)
print('First 5 rows (dx, dy, pen_lifted):')
print(s3[:5])

# TODO: Convert drawings[1] and drawings[2] to stroke-3 format.
#       Print the shape of each.
#       Are they the same length? Why or why not?
s3_1 = drawing_to_stroke3(drawings[1]['drawing'])
s3_2 = drawing_to_stroke3(drawings[2]['drawing'])
print(f"drawings[1] stroke-3 shape: {s3_1.shape}")
print(f"drawings[2] stroke-3 shape: {s3_2.shape}")

# OBSERVATIONS:
# They are not the same length. The total length corresponds to the total number of coordinates 
# recorded across all strokes in a sketch. One user might draw a fast, simple 20-point approximation, 
# while another might draw slowly, producing a higher density of path coordinates.



In [ ]:

# ==========================================
# 4. Normalisation
# ==========================================

def normalise_stroke3(stroke3):
    '''
    Normalise dx and dy to zero mean, unit std.
    The pen_lifted column (index 2) is left unchanged.
    '''
    s = stroke3.copy()
    coords = s[:, :2]
    mean = coords.mean(axis=0)
    std  = coords.std(axis=0) + 1e-8   # small constant to avoid dividing by zero
    s[:, :2] = (coords - mean) / std
    return s

s3      = drawing_to_stroke3(drawings[0]['drawing'])
s3_norm = normalise_stroke3(s3)
print('Before -- mean:', s3[:, :2].mean(axis=0).round(2), '  std:', s3[:, :2].std(axis=0).round(2))
print('After  -- mean:', s3_norm[:, :2].mean(axis=0).round(3), '  std:', s3_norm[:, :2].std(axis=0).round(3))

# TODO: What does the 1e-8 term protect against?
#       Describe a drawing that would have std = 0 in the x direction.

# OBSERVATIONS:
# 1. The 1e-8 term protects against ZeroDivisionError (division by zero) when calculating standard deviation.
# 2. A drawing would have std = 0 in the X direction if it is a completely vertical straight line, 
#    meaning the pen never moves left or right along the X-axis (all dx values are exactly 0).



In [ ]:

# ==========================================
# 5. Sequence Length Distribution
# ==========================================

lengths = []
for d in drawings[:2000]:
    s3 = drawing_to_stroke3(d['drawing'])
    lengths.append(len(s3))
lengths = np.array(lengths)

print(f'Mean   : {lengths.mean():.1f}')
print(f'Max    : {lengths.max()}')
print(f'95th % : {np.percentile(lengths, 95):.0f}')

# TODO: What percentage of drawings are longer than 200 steps?
#       Is 200 a reasonable cutoff for this category?
longer_than_200 = np.mean(lengths > 200) * 100
print(f"Percentage of drawings > 200 steps: {longer_than_200:.2f}%")

# OBSERVATIONS:
# Since the 95th percentile is well below 200, only a very tiny fraction of drawings (less than 1%) 
# exceed 200 steps. Therefore, 200 is an extremely reasonable cutoff value for truncation, 
# as it preserves the complete drawing data for over 99% of the dataset samples.



In [ ]:

# ==========================================
# 6. PyTorch Dataset
# ==========================================

class QuickDrawDataset(Dataset):
    def __init__(self, path, max_len=200, max_samples=5000):
        '''
        path        : path to .ndjson file
        max_len     : truncate sequences longer than this
        max_samples : load at most this many drawings
        '''
        self.samples = []
        with open(path) as f:
            for i, line in enumerate(f):
                if i >= max_samples:
                    break
                d  = json.loads(line)
                s3 = drawing_to_stroke3(d['drawing'])
                s3 = normalise_stroke3(s3)
                if len(s3) > max_len:
                    s3 = s3[:max_len]
                self.samples.append(torch.tensor(s3, dtype=torch.float32))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

dataset = QuickDrawDataset('data/cat.ndjson', max_len=200, max_samples=2000)
print(f'Dataset size       : {len(dataset)}')
print(f'First sample shape : {dataset[0].shape}')
print(f'First 4 rows:\n{dataset[0][:4]}')

loader = DataLoader(dataset, batch_size=1, shuffle=True)
batch  = next(iter(loader))
print('Batch shape:', batch.shape)   # (1, seq_len, 3)

# TODO: Why is batch_size=1 limiting for training?
#       What problem arises with batch_size=32 when sequences have different lengths?
#       (We will solve this with padding in Week 3.)

# OBSERVATIONS:
# 1. batch_size=1 is limiting because it doesn't leverage the parallel matrix multiplication 
#    capabilities of GPUs, leading to very slow training. It also makes gradient updates 
#    highly noisy and unstable.
# 2. When batch_size=32, PyTorch requires all sequences within that batch to stack into a 
#    uniform multidimensional tensor (e.g., shape 32 x Seq_Len x 3). If sequences have different 
#    lengths, PyTorch will raise an error because it cannot form a rigid rectangular matrix 
#    out of jagged rows without explicit padding.